## Vectorization with `vmap`

## The Batching Problem

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

In [2]:
def linear_single(w, b, x):
    return w @ x + b

In [3]:
w = jnp.ones((3, 4))
b = jnp.zeros(3)
x_single = jnp.ones(4)

In [4]:
print(f"Single output: {linear_single(w, b, x_single)}")
print(f"Output shape: {linear_single(w, b, x_single).shape}")

Single output: [4. 4. 4.]
Output shape: (3,)


In [5]:
X_batch = jnp.ones((8, 4))

results_loop = jnp.stack([linear_single(w, b, x) for x in X_batch])
print(f"Output shape: {results_loop.shape}")

Output shape: (8, 3)


## `vmap`: Write Once, Batch Automatically

In [6]:
# this runs linear_single(w, b, X_batch[i, :]) for every i
batched_linear = jax.vmap(linear_single, in_axes=(None, None, 0))

results_vmap = batched_linear(w, b, X_batch)
print(f"Output shape: {results_vmap.shape}")
print(jnp.allclose(results_vmap, results_loop))

Output shape: (8, 3)
True


## `in_axes` in Depth

In [7]:
def total(v):
    return jnp.sum(v)

In [8]:
M = jnp.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print(f"M: {M}")

M: [[1. 2. 3.]
 [4. 5. 6.]]


In [9]:
# total(M[i, :]) for every i
print(f"in_axes=0 - > {jax.vmap(total, in_axes=0)(M)}")

# total(M[:, j]) for every j
print(f"in_axes=1 - > {jax.vmap(total, in_axes=1)(M)}")

in_axes=0 - > [ 6. 15.]
in_axes=1 - > [5. 7. 9.]


In [10]:
print("=== unbatched ===")
print(jax.make_jaxpr(total)(jnp.ones(3)))

print("=== in_axes=0 ===")
print(jax.make_jaxpr(jax.vmap(total, in_axes=0))(M))

print("=== in_axes=1 ===")
print(jax.make_jaxpr(jax.vmap(total, in_axes=1))(M))

=== unbatched ===
{ lambda ; a:f32[3]. let
    b:f32[] = reduce_sum[axes=(0,) out_sharding=None] a
  in (b,) }
=== in_axes=0 ===
{ lambda ; a:f32[2,3]. let
    b:f32[2] = reduce_sum[axes=(np.int64(1),) out_sharding=None] a
  in (b,) }
=== in_axes=1 ===
{ lambda ; a:f32[2,3]. let
    b:f32[3] = reduce_sum[axes=(np.int64(0),) out_sharding=None] a
  in (b,) }


## `out_axes`: Controlling the Output Batch Dimension

In [11]:
def powers(x):
    return jnp.stack([x, x**2, x**3])

In [12]:
x = jnp.array([2.0, 3.0])

print(f"out_axes=0 - > {jax.vmap(powers, out_axes=0)(x)}")
print()
print(f"out_axes=1 - > {jax.vmap(powers, out_axes=1)(x)}")

out_axes=0 - > [[ 2.  4.  8.]
 [ 3.  9. 27.]]

out_axes=1 - > [[ 2.  3.]
 [ 4.  9.]
 [ 8. 27.]]


In [13]:
print("=== out_axes=0 ===")
print(jax.make_jaxpr(jax.vmap(powers, out_axes=0))(x))

print("=== out_axes=1 ===")
print(jax.make_jaxpr(jax.vmap(powers, out_axes=1))(x))

print("=== out_axes=0, then .T by hand ===")
print(jax.make_jaxpr(lambda X: jax.vmap(powers, out_axes=0)(X).T)(x))

=== out_axes=0 ===
{ lambda ; a:f32[2]. let
    b:f32[2] = integer_pow[y=2] a
    c:f32[2] = integer_pow[y=3] a
    d:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] a
    e:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] b
    f:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] c
    g:f32[2,3] = concatenate[dimension=1] d e f
  in (g,) }
=== out_axes=1 ===
{ lambda ; a:f32[2]. let
    b:f32[2] = integer_pow[y=2] a
    c:f32[2] = integer_pow[y=3] a
    d:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] a
    e:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] b
    f:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] c
    g:f32[2,3] = concatenate[dimension=1] d e f
    h:f32[3,2] = transpose[permutation=(1, 0)] g
  in (h,) }
=== out_axes=0, then .T by hand ===
{ lambda ; a:f32[2]. let
    b:f32[2] = integer_pow[y=2] a
    c:f32[2] = integer_pow[y=3] a
    d:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] a
    e:f32[2,1] = broadcast_in_dim[bro

## Per-Sample Gradients

In [14]:
def loss_single(params, x, y):
    pred = params["w"] @ x + params["b"]
    return jnp.sum((pred - y) ** 2)

In [15]:
key = jrandom.PRNGKey(0)

k_w, k_x, k_y = jrandom.split(key, 3)

params = {
    "w": jrandom.normal(k_w, (2, 3)),
    "b": jnp.zeros(2),
}

X = jrandom.normal(k_x, (8, 3))
y = jrandom.normal(k_y, (8, 2))

In [16]:
per_sample_grads = jax.vmap(jax.grad(loss_single), in_axes=(None, 0, 0))(params, X, y)
# print(f"Per-sample gradients: {per_sample_grads}")
print("Per-sample grad shapes:")
for k, v in per_sample_grads.items():
    print(f"   {k}: {v.shape}")

Per-sample grad shapes:
   b: (8, 2)
   w: (8, 2, 3)


## Pairwise Operations with Nested `vmap`

In [17]:
def squared_distance(x, y):
    diff = x - y
    return jnp.dot(diff, diff)

In [18]:
A = jnp.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
B = jnp.array([[0.0, 0.0], [2.0, 2.0]])

In [19]:
# Compute all pairwise distances: results[i, j] = dist(A[i], B[j])
pairwise = jax.vmap(jax.vmap(squared_distance, in_axes=(None, 0)), in_axes=(0, None))(
    A, B
)
print(f"pairwise: {pairwise}")

pairwise: [[1. 5.]
 [1. 5.]
 [2. 2.]]


In [20]:
jax.make_jaxpr(
    jax.vmap(jax.vmap(squared_distance, in_axes=(None, 0)), in_axes=(0, None))
)(A, B)

{ lambda ; a:f32[3,2] b:f32[2,2]. let
    c:f32[3,1,2] = broadcast_in_dim[broadcast_dimensions=(0, np.int64(2))] a
    d:f32[1,2,2] = broadcast_in_dim[
      broadcast_dimensions=(np.int64(1), np.int64(2))
    ] b
    e:f32[3,2,2] = sub c d
    f:f32[3,2] = dot_general[
      dimension_numbers=(([2], [2]), ([0, 1], [0, 1]))
      preferred_element_type=float32
    ] e e
  in (f,) }

## `jit(vmap(grad(...)))`: The Full Pattern

In [24]:
@jax.jit
def batched_train_step(params, X, y):
    per_grads = jax.vmap(jax.grad(loss_single), in_axes=(None, 0, 0))(params, X, y)
    mean_grads = jax.tree_util.tree_map(lambda g: jnp.mean(g, axis=0), per_grads)
    new_params = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, params, mean_grads)
    total_loss = jnp.mean(jax.vmap(loss_single, in_axes=(None, 0, 0))(params, X, y))
    return new_params, total_loss

In [27]:
losses = []
for _ in range(100):
    params, loss = batched_train_step(params, X, y)
    losses.append(float(loss))

In [28]:
losses

[2.440074920654297,
 2.4050891399383545,
 2.3713185787200928,
 2.3386993408203125,
 2.3071722984313965,
 2.276684284210205,
 2.2471847534179688,
 2.2186279296875,
 2.1909706592559814,
 2.1641740798950195,
 2.138200283050537,
 2.1130151748657227,
 2.0885868072509766,
 2.0648841857910156,
 2.041879653930664,
 2.0195460319519043,
 1.9978585243225098,
 1.9767931699752808,
 1.9563275575637817,
 1.9364402294158936,
 1.9171111583709717,
 1.8983211517333984,
 1.880051851272583,
 1.8622857332229614,
 1.8450062274932861,
 1.8281978368759155,
 1.811844825744629,
 1.7959330081939697,
 1.7804484367370605,
 1.7653776407241821,
 1.7507078647613525,
 1.736426591873169,
 1.722522497177124,
 1.7089838981628418,
 1.6957995891571045,
 1.6829594373703003,
 1.6704529523849487,
 1.6582705974578857,
 1.6464025974273682,
 1.6348401308059692,
 1.623574137687683,
 1.6125962734222412,
 1.6018980741500854,
 1.591471791267395,
 1.5813096761703491,
 1.571404218673706,
 1.5617485046386719,
 1.5523350238800049,
 1.543

In [37]:
jax.make_jaxpr(batched_train_step)(params, X, y)

{ lambda ; a:f32[2] b:f32[2,3] c:f32[8,3] d:f32[8,2]. let
    e:f32[2] f:f32[2,3] g:f32[] = jit[
      name=batched_train_step
      jaxpr={ lambda ; a:f32[2] b:f32[2,3] c:f32[8,3] d:f32[8,2]. let
          h:f32[2,8] = dot_general[
            dimension_numbers=(([1], [1]), ([], []))
            preferred_element_type=float32
          ] b c
          i:f32[8,2] = transpose[permutation=(1, 0)] h
          j:f32[1,2] = broadcast_in_dim[broadcast_dimensions=(np.int64(1),)] a
          k:f32[8,2] = add i j
          l:f32[8,2] = sub k d
          m:f32[8,2] = integer_pow[y=2] l
          n:f32[8,2] = integer_pow[y=1] l
          o:f32[8,2] = mul 2.0:f32[] n
          _:f32[8] = reduce_sum[axes=(np.int64(1),) out_sharding=None] m
          p:f32[2] = broadcast_in_dim 1.0:f32[]
          q:f32[1,2] = broadcast_in_dim[broadcast_dimensions=(np.int64(1),)] p
          r:f32[8,2] = mul q o
          s:f32[8,2,3] = dot_general[
            dimension_numbers=(([], []), ([0], [0]))
            pr